In [2]:
import numpy as np

In [17]:
rng = np.random.default_rng(seed=1) # Returns a Generator object wrapping the default bit generator (PCG64)

In [18]:
# Samplings are then performed via rng, not via np.random.
rng.normal(loc=0, scale=1, size=5)

array([ 0.34558419,  0.82161814,  0.33043708, -1.30315723,  0.90535587])

# Distributions

## Normal
formula: x ~ N (μ, σ**2)
- loc → mean (μ)
- scale → standard deviation (σ)

## Uniform
formula: x ~ u (a, b)
- Every value between a and b is equally likely.

## Binomial
formula: x ~ Binomial (n, p)
- Models number of successes in n independent Bernoulli trials.
- n: number of trials
- p: probability of success

In [19]:
# Let's stimulate a mini-detaset of reaction times.

n_trials = 100
mean_rt= 600
sd_rt = 50

rt = rng.normal(loc=mean_rt, scale=sd_rt, size= n_trials)

# Remove impossible negative values
rt = np.clip(rt, a_min=100, a_max=None)

In [20]:
# Better alternative is lognormal
rt = rng.lognormal(mean=6.3, sigma=0.15, size=n_trials)

Note: common mistakes about seeds
- Calling np.random.seed() inside functions.
- Re-seeding repeatedly (destroys random structure).
- Mixing legacy and new APIs.
- Forgetting to store seed in simulation logs.

In [21]:
# Let's create two experimental conditions: one with an easy task and one with a hard task (reaction time and accuracy will be different).

# Condition A
rt_A = rng.normal(loc=550, scale=70, size=n_trials)

# Condition B
rt_B = rng.normal(loc=650, scale=90, size=n_trials)

acc_A = rng.binomial(n=1, p=0.90, size=n_trials)
acc_B = rng.binomial(n=1, p=0.75, size=n_trials)

# Now mean rt_A < mean rt_B, and mean acc_A > mean acc_B.

np.nan
- It is an IEEE-754 floating-point, not a number value.
- It represents undefined or unrepresentable results (e.g., 0/0).
- It is not equal to anything, including itself.

In [22]:
print(np.nan == np.nan)
print(np.nan is np.nan)

False
True


In [23]:
# Note: any arithmetic operation involving NaN propagates NaN.
nan_arr = np.array([1,2,np.nan,4])
nan_arr+1

array([ 2.,  3., nan,  5.])

In [24]:
# We use np.isnan for detection, because NaN != NaN.
mask = np.isnan(nan_arr)
print(mask)

[False False  True False]


In [27]:
# Standard aggregation functions treat NaN as a contaminant, so we should use nan-safe functinos.
print(np.mean(nan_arr), np.nanmean(nan_arr))
print(np.sum(nan_arr), np.nansum(nan_arr))
print(np.std(nan_arr), np.nanstd(nan_arr))
print(np.var(nan_arr), np.nanvar(nan_arr))

nan 2.3333333333333335
nan 7.0
nan 1.247219128924647
nan 1.5555555555555554


Note: NaN handling is not statistical modeling; it is computational masking.

In [28]:
print(0.1 + 0.2)

0.30000000000000004


## Floating-Point Precision  
float64 structure:
- 1 sign bit
- 11 exponent bits
- 52 fraction bits
≈ 15–17 decimal digits precision

Finite precision implies:
1. Rounding error
2. Non-associativity of addition
3. Representation error

In [30]:
# Representation error
0.1+0.2

0.30000000000000004

In [31]:
# 0.1 and 0.2 are not exactly representable in binary.

In [39]:
# Machine epsilon

np.finfo(float).eps # eps is the smallest number.

np.float64(2.220446049250313e-16)

In [40]:
# Loss of significance (catastrophic cancellation)

num1 = 1e16
num2 = num1 + 1

print(num2 - num1) 

0.0


Note: the +1 is below precision resolution relative to 1e16, so it disappears.

In [42]:
# Non-associativity

num3 = 1e16
num4 = -1e16
num5 = 1.0

print((num3 + num4) + num5)
print(num3 + (num4 + num5))

1.0
0.0


Note: floating-point addition is not associative.

In [45]:
# Overflow: occurs when result exceeds maximum representable float; could be inf or -inf.
np.exp(1000)

C:\Users\Erfan\AppData\Local\Temp\ipykernel_10916\1506384762.py:2: RuntimeWarning: overflow encountered in exp
  np.exp(1000)


np.float64(inf)

In [46]:
# Underflow: occurs when result is too small in magnitude to represent; could be 0.0 or subnormal numbers (reduced precision).
np.exp(-1000)

np.float64(0.0)

In [48]:
# The limitations are:
print(np.finfo(float).max)
print(np.finfo(float).tiny)

1.7976931348623157e+308
2.2250738585072014e-308


In [49]:
# Operations involving inf can lead to NaN:
np.inf - np.inf

nan

In [51]:
# Let's build a missing data dataset.

np.random.seed(1)
data = np.random.normal(loc=100,scale=15,size=1000)

# Adding 10% missing:
missing_mask = np.random.rand(1000) < 0.10
data_with_nan = data.copy()
data_with_nan[missing_mask] = np.nan

# Let's verify.
np.isnan(data_with_nan).sum()

np.int64(88)

In [56]:
# Listwise deletion
deleted = data_with_nan[~np.isnan(data_with_nan)]
mean_deleted = np.mean(deleted)
print(mean_deleted)

100.78605564104548


In [57]:
# Mean imputation
imputed = data_with_nan.copy()
mean_value = np.nanmean(imputed)
imputed[np.isnan(imputed)] = mean_value
mean_imputed = np.mean(imputed)
print(mean_imputed)

100.7860556410455


In [59]:
true_mean = np.mean(data)
print(true_mean)

100.58218714239403


Deletion:
- Unbiased if data are MCAR (Missing Completely At Random).
- Reduces sample size → increases variance.

Mean imputation:
- Preserves sample size.
- Biases variance downward.
- Distorts covariance structure.
- Underestimates uncertainty.

Note: mean imputation is not statistically principled; it treats imputed values as known.  
For rigorous modeling use multiple imputation or likelihood-based methods.

In [60]:
# Simulate large mean:
np.random.seed(0)
x = 1e8 + np.random.normal(0, 1, 1000000)

# Variance naive formula: Var(X) = E[X**2] − (E[X])**2

# Naive computation:
var_naive = np.mean(x**2) - np.mean(x)**2

# Stable computation:
var_stable = np.var(x)

# Compare:
print(var_naive, var_stable)

0.0 0.9998426582217603


Note: there is numerical instability in NumPy. Naive computation may be inaccurate due to subtracting large nearly-equal numbers.

In [4]:
a1 = np.array([[1, 1],
              [1, 1.0000000001]])

np.linalg.inv(a1)

array([[ 9.99999917e+09, -9.99999917e+09],
       [-9.99999917e+09,  9.99999917e+09]])

In [11]:
np.linalg.cond(a1)

np.float64(39999947702.450424)

Note:
1. Use NumPy’s built-in statistical functions.
2. Avoid subtracting nearly equal large numbers.
3. Avoid explicit matrix inversion.
4. Use float64 unless you have a strong reason not to.
5. Center variables before regression.
6. Inspect condition numbers in linear models.